# 🎮 Reverse Survivor: The Director — Обучение Глубокой Нейросети в Google Colab

### ⚙️ Настройки среды Colab:
1. В верхнем меню выберите: **Среда выполнения (Runtime) -> Сменить среду выполнения (Change runtime type)**.
2. **Аппаратный ускоритель**: выберите **`ЦП` (CPU)**.
   *(Почему ЦП? Наш движок на чистом Rust + Rayon эмулирует физику матчей на ядрах процессора. Видеокарта T4 не задействуется и только потратит дневные лимиты Colab)*.
3. *(Опционально)*: в форме ОЗУ выберите **`Высокий объем ОЗУ` (High-RAM)**, если доступно.

---
### 🚀 Как запустить обучение в 1 клик:
- **Вариант А (самый удобный)**: В левой панели Colab откройте вкладку 📁 **Файлы (Files)** и просто **перетащите мышкой** файл `Anti_VS_Colab.zip` со своего компьютера.
- **Вариант Б**: Просто нажмите `Run` (▶) на ячейке ниже — появится кнопка **«Выбрать файлы»** для загрузки архива.

In [ ]:
# ==============================================================================
# ⚡ [ВСЁ-В-ОДНОМ] Установка, распаковка, обучение (150 сенсоров) и скачивание
# ==============================================================================
import os, shutil, zipfile
from google.colab import files

# 1. Проверяем или загружаем архив проекта
zip_name = 'Anti_VS_Colab.zip'
if not os.path.exists(f'/content/{zip_name}'):
    print("📦 Файл Anti_VS_Colab.zip не найден в /content/. Загрузите его через кнопку ниже:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            zip_name = fn
            break

# 2. Распаковываем проект в /content/Anti_VS
work_dir = '/content/Anti_VS'
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
os.makedirs(work_dir, exist_ok=True)

print(f"📂 Распаковка {zip_name} в {work_dir}...")
with zipfile.ZipFile(f'/content/{zip_name}', 'r') as zip_ref:
    for member in zip_ref.infolist():
        clean_path = member.filename.replace('\\', '/').lstrip('/')
        target_file = os.path.join(work_dir, clean_path)
        if member.is_dir() or clean_path.endswith('/'):
            os.makedirs(target_file, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target_file), exist_ok=True)
            with open(target_file, 'wb') as f:
                f.write(zip_ref.read(member.filename))
print("✅ Проект успешно распакован!")

# 3. Проверка и быстрая установка компилятора Rust
if not os.path.exists('/root/.cargo/bin/cargo'):
    print("🦀 Установка компилятора Rust (Cargo & rustc)...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!rustc --version

# 4. Переход в папку проекта и запуск турнира обучения
%cd /content/Anti_VS

# Количество поколений и размер популяции (например: 20 поколений, 16 кандидатов)
# Все ядра процессора Colab будут загружены на 100% благодаря Rayon!
!cargo run --release -- --train 20 16

# 5. Автоматическая упаковка и скачивание обученных моделей в браузер
models_dir = '/content/Anti_VS/models'
out_zip = '/content/Anti_VS_Trained_Models.zip'
if os.path.exists(models_dir):
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, m_files in os.walk(models_dir):
            for mf in m_files:
                full_p = os.path.join(root, mf)
                rel_p = os.path.relpath(full_p, models_dir)
                zf.write(full_p, rel_p)
    print("\n🏆 Обучение завершено! Скачиваем архив с новыми моделями...")
    files.download(out_zip)
else:
    print("⚠️ Папка models не найдена. Проверьте вывод обучения выше.")


--- 
### 🔄 Пошаговый режим (если хотите перезапустить только обучение без распаковки)

In [ ]:
# Запуск дополнительного раунда обучения (продолжение тренировки)
%cd /content/Anti_VS
!cargo run --release -- --train 15 16

In [ ]:
# Ручное скачивание архива с моделями в любой момент
import zipfile, os
from google.colab import files

models_dir = '/content/Anti_VS/models'
out_zip = '/content/Anti_VS_Trained_Models.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, m_files in os.walk(models_dir):
        for mf in m_files:
            full_p = os.path.join(root, mf)
            zf.write(full_p, os.path.relpath(full_p, models_dir))

files.download(out_zip)
print("✅ Скачивание запущено!")